In [22]:
from src.data_loader import DataLoader
from src.power_station import PowerStation
from src.load_forecaster import LoadForecaster
from src.outage_analyzer import OutageAnalyzer
from src.risk_model import RiskModel

loader = DataLoader("live_grid_load.csv", "data/raw/weather_hudson_station.csv", "data/raw/project_electrical_outages.csv")
merged = loader.merge_all()

station = PowerStation("Hudson Substation", 40.728, -74.078, rated_capacity=85)
station.load_history = merged['Load_Percent'].tolist()

forecaster = LoadForecaster(merged)
forecast_hour = forecaster.forecast_next_hour()

outages = loader.load_outage_data()
out_analyzer = OutageAnalyzer(outages)
out_prob = out_analyzer.compute_outage_probability(merged['Timestamp'].iloc[-1])

risk_engine = RiskModel(station.rated_capacity)
risk_score = risk_engine.compute_risk_score(forecast_hour, out_prob, merged['temperature_C'].iloc[-1])
risk_level = risk_engine.classify(risk_score)

print("Next-hour forecast:", forecast_hour)
print("Outage probability:", out_prob)
print("Risk score:", risk_score, "=>", risk_level)

Next-hour forecast: 65.74343056902627
Outage probability: 0.03749043611323642
Risk score: 0.47119388363320047 => Moderate
